In [11]:
import neptune
import pandas as pd
import numpy as np

# Edit run meta info

In [12]:
# # Reopen by run ID (from sys/id)
# run = neptune.init_run(
#     with_id="DNAT-274",       # e.g. your run id
#     project="amar-mesic/dna-thesis",
#     mode="sync",         # important, otherwise it's read-only
#     api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
# )

# # Add or overwrite metadata
# run["meta/experiment"] = "Synth_data-Genotype_aware-SPS"
# run["meta/model"] = "Amar-DNANet_Lite"
# run["meta/dataset"] = "proved_it"
# run["meta/seed"] = 43
# run["meta/fold"] = 1

# # Add tags too (handy in UI)
# run["sys/tags"].add(["exp:Synth_data-Genotype_aware-SPS", "model:Amar-DNANet_Lite", "dataset:proved_it", "seed:43", "fold:1"])

# # Always stop after edits
# run.stop()

In [13]:
# connect to your Neptune project
project = neptune.init_project(
    "amar-mesic/dna-thesis",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
)

# fetch runs (maybe filtered by tag or name)
runs_table = project.fetch_runs_table(columns=[
    "sys/name", 
    "meta/experiment", "meta/model", "meta/dataset", "meta/seed", "meta/fold",
    "test/pixel_f1", "test/pixel_precision", "test/pixel_recall",
    "test/allele_f1", "test/allele_precision", "test/allele_recall"
])

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/


In [14]:
runs_table

In [15]:
# %pip install -U ipywidgets

# get dataframe
df = runs_table.to_pandas()

# rename to cleaner columns
df = df.rename(columns={
    "meta/experiment": "experiment",
    "meta/model": "model",
    "meta/dataset": "dataset",
    "meta/seed": "seed",
    "meta/fold": "fold",
})

df

Fetching table...: 0 [00:00, ?/s]

,sys/creation_time,sys/id,sys/name,dataset,experiment,fold,model,seed,test/allele_f1,test/allele_precision,test/allele_recall,test/pixel_f1,test/pixel_precision,test/pixel_recall
0,2025-09-09 11:03:04.944,DNAT-391,670+长-1正真:0正假-考试倍1-预：训练，缩放-种4,proved_it_+_synth_1:0,Synth_data-Genotype_aware-SPS,1.0,Amar-DNANet_Advanced,4.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-09 10:45:10.990,DNAT-390,670+长-1正真:0正假-考试倍1-预：训练，缩放-种3,proved_it_+_synth_1:0,Synth_data-Genotype_aware-SPS,1.0,Amar-DNANet_Advanced,3.0,0.8257,0.8992,0.7634,0.8517,0.868,0.836
2,2025-09-09 10:34:51.353,DNAT-389,670+长-1正真:4正假-考试倍0-预：训练，缩放-种1,proved_it_+_synth_1:4,Synth_data-Genotype_aware-SPS,0.0,Amar-DNANet_Advanced,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-09-09 10:27:07.422,DNAT-388,670+长-1正真:0正假-考试倍1-预：训练，缩放-种2,proved_it_+_synth_1:0,Synth_data-Genotype_aware-SPS,1.0,Amar-DNANet_Advanced,2.0,0.8257,0.8992,0.7634,0.8517,0.868,0.836
4,2025-09-09 10:08:51.455,DNAT-387,670+长-1正真:0正假-考试倍1-预：训练，缩放-种1,proved_it_+_synth_1:0,Synth_data-Genotype_aware-SPS,1.0,Amar-DNANet_Advanced,1.0,0.8257,0.8992,0.7634,0.8517,0.868,0.836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,2025-07-02 08:10:54.198,DNAT-63,最好U网-100真:0假-早停-lr0.01-用exp调度0.98-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,2025-07-01 18:04:27.351,DNAT-62,最好U网-100真:0假-早停-lr0.01-用cyc调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
169,2025-07-01 17:07:54.342,DNAT-60,最好U网-100真:0假-早停-lr0.01-用cos调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
170,2025-07-01 14:13:53.630,DNAT-54,最好U网-100真:0假-早停-lr0.01-不调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# melt to long format (so you can aggregate all test uniformly)
df_long = df.melt(
    id_vars=["experiment", "model", "dataset", "seed", "fold"],
    value_vars=[
        "test/pixel_f1", "test/pixel_precision", "test/pixel_recall",
        "test/allele_f1", "test/allele_precision", "test/allele_recall"
    ],
    var_name="metric",
    value_name="value"
)
df_long

,experiment,model,dataset,seed,fold,metric,value
0,Synth_data-Genotype_aware-SPS,Amar-DNANet_Advanced,proved_it_+_synth_1:0,4.0,1.0,test/pixel_f1,NaN
1,Synth_data-Genotype_aware-SPS,Amar-DNANet_Advanced,proved_it_+_synth_1:0,3.0,1.0,test/pixel_f1,0.8517
2,Synth_data-Genotype_aware-SPS,Amar-DNANet_Advanced,proved_it_+_synth_1:4,1.0,0.0,test/pixel_f1,NaN
3,Synth_data-Genotype_aware-SPS,Amar-DNANet_Advanced,proved_it_+_synth_1:0,2.0,1.0,test/pixel_f1,0.8517
4,Synth_data-Genotype_aware-SPS,Amar-DNANet_Advanced,proved_it_+_synth_1:0,1.0,1.0,test/pixel_f1,0.8517
...,...,...,...,...,...,...,...
1027,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
1028,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
1029,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
1030,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN


In [17]:
# group + aggregate
summary = (
    df_long
    .groupby(["experiment", "model", "dataset", "metric"])
    .agg(mean=("value", "mean"), std=("value", "std"), 
         min=("value", "min"), max=("value", "max"), n=("value", "count"))
    .reset_index()
)

summary

,experiment,model,dataset,metric,mean,std,min,max,n
0,AT_Baseline,AT_75,proved_it,test/allele_f1,0.758167,0.004325,0.75360,0.76220,3
1,AT_Baseline,AT_75,proved_it,test/allele_precision,0.799100,0.010400,0.78710,0.80550,3
2,AT_Baseline,AT_75,proved_it,test/allele_recall,0.721433,0.012287,0.70810,0.73230,3
3,AT_Baseline,AT_75,proved_it,test/pixel_f1,0.148100,0.002835,0.14490,0.15030,3
4,AT_Baseline,AT_75,proved_it,test/pixel_precision,0.080637,0.001725,0.07868,0.08194,3
5,AT_Baseline,AT_75,proved_it,test/pixel_recall,0.906800,0.006391,0.89960,0.91180,3
6,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_f1,0.792000,0.023909,0.77230,0.81860,3
7,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_precision,0.918100,0.020419,0.90150,0.94090,3
8,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_recall,0.696500,0.027420,0.66970,0.72450,3
9,DNANet_Baseline,DNANet_Pretrained,proved_it,test/pixel_f1,0.820933,0.046136,0.79360,0.87420,3


In [18]:
def format_row(summary, experiment, model, dataset):
    # metric order
    metrics = [
        "test/allele_f1",
        "test/allele_precision",
        "test/allele_recall",
        "test/pixel_f1",
        "test/pixel_precision",
        "test/pixel_recall",
    ]
    
    # filter for the experiment
    df = summary[
        (summary["experiment"] == experiment)
        & (summary["model"] == model)
        & (summary["dataset"] == dataset)
    ]
    
    values = []
    for m in metrics:
        row = df[df["metric"] == m]
        if row.empty:
            values.append("–")  # no data
        else:
            mean = row["mean"].iloc[0]
            std = row["std"].iloc[0]
            if np.isnan(std) or std == 0:
                values.append(f"{mean:.3f}")
            else:
                # format with 2 sig digits for std
                std_fmt = f"{std:.2g}"
                values.append(f"${mean:.3f}\\pm{std_fmt}$")
    
    return " & ".join(values) + " \\\\"


In [19]:
# Example usage
latex_row = format_row(summary, 
                       experiment="Synth_data-Genotype_aware-SPS",
                       model="Amar-DNANet_Advanced",
                       dataset="proved_it_+_synth_1:0")

print(latex_row)

$0.823\pm0.0023$ & $0.923\pm0.023$ & $0.744\pm0.019$ & $0.854\pm0.0025$ & $0.882\pm0.013$ & $0.828\pm0.0072$ \\


In [21]:
[1,2,3,4,5][2:]

[3, 4, 5]